# Проверка PaddleOCR, MinerU и Unlimited OCR

Ноутбук запускает три независимых backend'а на одном файле и собирает время, статус и артефакты.

- **PaddleOCR** — дефолт, работает локально на CPU/GPU.
- **MinerU** — запускается через CLI и возвращает Markdown/JSON.
- **Unlimited OCR** — прямой Transformers inference на NVIDIA GPU.

Тяжёлые зависимости специально не устанавливаются автоматически. Используйте отдельные окружения для production; этот notebook предназначен только для сравнения.

> Перед запуском задайте `INPUT_PATH`. Для честного сравнения начните с PNG/JPG или PDF на 1–3 страницы. Unlimited OCR требует NVIDIA CUDA; на Mac его ячейка будет корректно пропущена.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

INPUT_PATH = Path("samples/document.pdf")  # <- замените на свой PDF/JPG/PNG
OUTPUT_ROOT = Path("benchmark_outputs")
LANG = "ru"  # PaddleOCR: ru, en и т.д.

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS: dict[str, dict[str, Any]] = {}


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def record(engine: str, started: float, status: str, **details: Any) -> None:
    RESULTS[engine] = {
        "status": status,
        "seconds": round(time.perf_counter() - started, 3),
        **details,
    }


print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "input": str(INPUT_PATH),
    "input_exists": INPUT_PATH.exists(),
    "paddleocr_installed": has_module("paddleocr"),
    "mineru_cli": shutil.which("mineru"),
    "torch_installed": has_module("torch"),
})

## Установка (выполняйте только нужные команды)

Рекомендуется отдельное виртуальное окружение для каждого тяжёлого движка.

```bash
# Базовые утилиты notebook
pip install jupyter pillow pymupdf pandas

# PaddleOCR (CPU; сверяйтесь с актуальной инструкцией PaddlePaddle для вашей ОС)
pip install paddlepaddle paddleocr

# MinerU
pip install "mineru[core]"
# CLI должен появиться как: mineru

# Unlimited OCR — только NVIDIA/CUDA; официально протестированные версии на момент создания notebook
pip install torch==2.10.0 torchvision==0.25.0 transformers==4.57.1 \
  Pillow==12.1.1 matplotlib==3.10.8 einops==0.8.2 addict==2.4.0 \
  easydict==1.13 pymupdf==1.27.2.2 psutil==7.2.2
```

На Apple Silicon используйте PaddleOCR и MinerU. Unlimited OCR сейчас ориентирован на NVIDIA CUDA и должен запускаться на отдельной GPU-машине.

In [ ]:
def prepare_page_images(path: Path, dpi: int = 200) -> list[Path]:
    """Возвращает исходное изображение или рендерит PDF постранично."""
    if not path.exists():
        raise FileNotFoundError(f"Укажите существующий INPUT_PATH, сейчас: {path}")
    if path.suffix.lower() != ".pdf":
        return [path]
    if not has_module("fitz"):
        raise RuntimeError("Для PDF установите pymupdf: pip install pymupdf")

    import fitz

    out_dir = OUTPUT_ROOT / "rendered_pages"
    out_dir.mkdir(parents=True, exist_ok=True)
    pages: list[Path] = []
    with fitz.open(path) as document:
        matrix = fitz.Matrix(dpi / 72, dpi / 72)
        for index, page in enumerate(document):
            output = out_dir / f"page_{index + 1:04d}.png"
            page.get_pixmap(matrix=matrix, alpha=False).save(output)
            pages.append(output)
    return pages


PAGE_IMAGES = prepare_page_images(INPUT_PATH)
print(f"Подготовлено страниц: {len(PAGE_IMAGES)}")
PAGE_IMAGES[:3]

In [ ]:
# PaddleOCR — дефолтный backend
started = time.perf_counter()
try:
    if not has_module("paddleocr"):
        raise RuntimeError("PaddleOCR не установлен; выполните pip install paddlepaddle paddleocr")

    from paddleocr import PaddleOCR

    try:
        # API PaddleOCR 3.x
        paddle = PaddleOCR(
            lang=LANG,
            use_doc_orientation_classify=True,
            use_doc_unwarping=False,
            use_textline_orientation=True,
        )
        raw_pages = [list(paddle.predict(str(page))) for page in PAGE_IMAGES]
    except TypeError:
        # Совместимость со старыми версиями
        paddle = PaddleOCR(lang=LANG, use_angle_cls=True)
        raw_pages = [paddle.ocr(str(page), cls=True) for page in PAGE_IMAGES]

    def make_jsonable(value: Any) -> Any:
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        if isinstance(value, dict):
            return {str(k): make_jsonable(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [make_jsonable(v) for v in value]
        for attr in ("json", "to_json", "to_dict"):
            candidate = getattr(value, attr, None)
            if candidate is not None:
                candidate = candidate() if callable(candidate) else candidate
                if isinstance(candidate, str):
                    try:
                        candidate = json.loads(candidate)
                    except json.JSONDecodeError:
                        return candidate
                return make_jsonable(candidate)
        return str(value)

    paddle_data = make_jsonable(raw_pages)
    paddle_dir = OUTPUT_ROOT / "paddleocr"
    paddle_dir.mkdir(exist_ok=True)
    (paddle_dir / "result.json").write_text(
        json.dumps(paddle_data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    # Работает с 3.x (rec_texts) и старым форматом [[bbox, (text, score)]].
    texts: list[str] = []
    def collect_text(value: Any) -> None:
        if isinstance(value, dict):
            if isinstance(value.get("rec_texts"), list):
                texts.extend(str(x) for x in value["rec_texts"])
            else:
                for child in value.values():
                    collect_text(child)
        elif isinstance(value, list):
            if len(value) == 2 and isinstance(value[1], (list, tuple)) and value[1] and isinstance(value[1][0], str):
                texts.append(value[1][0])
            else:
                for child in value:
                    collect_text(child)

    collect_text(paddle_data)
    paddle_text = "\n".join(texts)
    (paddle_dir / "result.txt").write_text(paddle_text, encoding="utf-8")
    record("PaddleOCR", started, "ok", pages=len(PAGE_IMAGES), chars=len(paddle_text), output=str(paddle_dir))
    print(paddle_text[:3000])
except Exception as exc:
    record("PaddleOCR", started, "error", error=f"{type(exc).__name__}: {exc}")
    print(RESULTS["PaddleOCR"])

In [ ]:
# MinerU — структурный backend через официальный CLI
MINERU_BACKEND = "pipeline"  # CPU-friendly; для GPU можно выбрать hybrid/vlm по документации
started = time.perf_counter()
try:
    mineru_executable = shutil.which("mineru")
    if not mineru_executable:
        raise RuntimeError("CLI mineru не найден. Установите MinerU в активное окружение.")

    mineru_dir = OUTPUT_ROOT / "mineru"
    mineru_dir.mkdir(exist_ok=True)
    command = [
        mineru_executable,
        "-p", str(INPUT_PATH.resolve()),
        "-o", str(mineru_dir.resolve()),
        "-b", MINERU_BACKEND,
    ]
    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
        timeout=3600,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr[-4000:] or completed.stdout[-4000:])

    markdown_files = sorted(mineru_dir.rglob("*.md"))
    markdown = "\n\n".join(path.read_text(encoding="utf-8") for path in markdown_files)
    record(
        "MinerU",
        started,
        "ok",
        backend=MINERU_BACKEND,
        chars=len(markdown),
        markdown_files=len(markdown_files),
        output=str(mineru_dir),
    )
    print(markdown[:3000] if markdown else f"Готово. Артефакты: {mineru_dir}")
except subprocess.TimeoutExpired:
    record("MinerU", started, "error", error="TimeoutExpired: превышен лимит 3600 сек")
    print(RESULTS["MinerU"])
except Exception as exc:
    record("MinerU", started, "error", error=f"{type(exc).__name__}: {exc}")
    print(RESULTS["MinerU"])